# MACE+Graph2Mat

This notebook will show you how to integrate a `MACE` model with `Graph2Mat` through the python API. Note that you can also use `MACE+Graph2Mat` through the Command Line Interface (CLI).

Prerequisites
-------------
Before reading this notebook, **make sure you have read the [notebook on computing a matrix](<./Computing a matrix.ipynb>) and [the notebook on batching](./Batching.ipynb)**, which introduce the basic concepts of `graph2mat` that we are going to assume are already known. Also **we will use exactly the same setup as in the batching notebook**, with the only difference that we will add target matrices to each structure.

In [1]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [2]:
import numpy as np
import pandas as pd
import torch

# To load plotly templates for sisl visualization
import sisl.viz

from e3nn import o3

from graph2mat import (
    BasisConfiguration,
    PointBasis,
    BasisTableWithEdges,
    MatrixDataProcessor,
)
from graph2mat.bindings.torch import TorchBasisMatrixDataset, TorchBasisMatrixData

from graph2mat.bindings.e3nn import E3nnGraph2Mat

from graph2mat.tools.viz import plot_basis_matrix

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


Generating a dataset
--------------------

We generate a dataset here just as we have done in the other notebooks.

In [3]:
# The basis
point_1 = PointBasis("A", R=2, basis="0e", basis_convention="spherical", matrix_role='row')  # "0e"
point_2 = PointBasis("A", R=2, basis="2x0e", basis_convention="spherical", matrix_role='col')
point_3 = PointBasis("B", R=5, basis="0e + 1o", basis_convention="spherical", matrix_role='row')
point_4 = PointBasis("B", R=5, basis="2x0e + 1o", basis_convention="spherical", matrix_role='col')


# The basis table.
table = BasisTableWithEdges([point_1, point_2, point_3, point_4])

# The data processor.
processor = MatrixDataProcessor(
    basis_table=table, symmetric_matrix=False,  # Matrix is not square
    sub_point_matrix=False
)

positions = np.array([[0, 0, 0], [6.0, 0, 0], [9, 0, 0]])

config1 = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions,
    basis=[point_1, point_2, point_3, point_4],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

config2 = BasisConfiguration(
    point_types=["B", "A", "B"],
    positions=positions,
    basis=[point_1, point_2, point_3, point_4],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

configs = [config1, config2]

dataset = TorchBasisMatrixDataset(configs, data_processor=processor)

from torch_geometric.loader import DataLoader

loader = DataLoader(dataset, batch_size=2)

data = next(iter(loader))

Defined row for type A with basis ((1, 0, 1),) and reach 2.
Defined col for type A with basis ((2, 0, 1),) and reach 2.
Defined row for type B with basis ((1, 0, 1), (1, 1, -1)) and reach 5.
Defined col for type B with basis ((2, 0, 1), (1, 1, -1)) and reach 5.
BasisTableWithEdges: is_square = False
BasisTableWithEdges: all matrix roles = ['row', 'col', 'row', 'col']
Basis sizes: [1 4]
Basis sizes: [2 5]
Row cutoff radii: [2 5]
Col cutoff radii: [2 5]
Max cutoff radius: [2 5]
In BasisTableWithEdges: 
self.edge_type == point_types_to_edge_types:
[[ 0  1]
 [-1  2]]
self.point_block_shape:
[[1 4]
 [2 5]]
self.point_block_size:
[ 2 20]
Row basis sizes: [1 4]
Col basis sizes: [2 5]
Point type to edge type:
[[ 0  1]
 [-1  2]]
Edge type to point types:
[[0 0]
 [0 1]
 [1 1]]
Edge block shape:
[[1 1 4]
 [2 5 5]]
Edge block shape inv:
[[1 4 4]
 [2 2 5]]
self.basis_table.R is an array: [2 5]
point_types: [0 1 0]
self.basis_table.R[point_types]: [2 5 2]
Cutoff: [1.9999 4.9999 1.9999]
In BasisMatri

Initializing a MACE model
-------------------------

We will now initialize a normal MACE model.

Note that you must have MACE installed, which you can do with:

```
pip install mace_torch
```

In [4]:
from mace.modules import MACE, RealAgnosticResidualInteractionBlock

num_interactions = 3
hidden_irreps = o3.Irreps("1x0e + 1x1o")

mace_model = MACE(
    r_max=10,
    num_bessel=10,
    num_polynomial_cutoff=10,
    max_ell=2,  # 1,
    interaction_cls=RealAgnosticResidualInteractionBlock,
    interaction_cls_first=RealAgnosticResidualInteractionBlock,
    num_interactions=num_interactions,
    num_elements=2,
    hidden_irreps=hidden_irreps,
    MLP_irreps=o3.Irreps("2x0e"),
    atomic_energies=torch.tensor([0, 0]),
    avg_num_neighbors=2,
    atomic_numbers=[0, 1],
    correlation=2,
    gate=None,
)

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/mace/modules/blocks.py:312: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(atomic_energies, dtype=torch.get_default_dtype()),
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

Now, we can pass our data through the mace model. MACE outputs many things, but we are just interested in the node features, which we can get from the `"node_feats"` key.

In [5]:
mace_output = mace_model(data)
mace_output["node_feats"]

tensor([[ 1.5062e-02,  0.0000e+00,  0.0000e+00, -4.3382e-03,  2.1096e-02,
          0.0000e+00,  0.0000e+00, -5.4341e-03, -2.6008e-02],
        [-2.5060e-01,  0.0000e+00,  0.0000e+00,  5.6106e-05, -3.0464e-02,
          0.0000e+00,  0.0000e+00, -4.1628e-04, -3.5715e-02],
        [ 1.5020e-02,  0.0000e+00,  0.0000e+00, -1.1716e-02,  2.1039e-02,
          0.0000e+00,  0.0000e+00, -1.8535e-02, -2.5963e-02],
        [-2.5058e-01,  0.0000e+00,  0.0000e+00, -8.1687e-05, -3.0457e-02,
          0.0000e+00,  0.0000e+00,  1.6378e-04, -3.5703e-02],
        [ 1.6150e-02,  0.0000e+00,  0.0000e+00,  1.6057e-02,  2.2623e-02,
          0.0000e+00,  0.0000e+00,  2.3973e-02, -2.7921e-02],
        [-2.5057e-01,  0.0000e+00,  0.0000e+00,  2.5577e-05, -3.0460e-02,
          0.0000e+00,  0.0000e+00,  3.1330e-04, -3.5710e-02]],
       grad_fn=<CatBackward0>)

Our `Graph2Mat` model will take these node features and convert them to a matrix. Therefore we need to know what its irreps are, and then initialize the `Graph2Mat` module.

In [6]:
# MACE outputs as node features the hidden irreps for each interaction, except
# in the last interaction, where it computes just scalar features.
mace_out_irreps = hidden_irreps * (num_interactions - 1) + str(hidden_irreps[0])

# Initialize the matrix model with this information
matrix_model = E3nnGraph2Mat(
    unique_basis=table,
    irreps=dict(node_feats_irreps=mace_out_irreps),
    symmetric=False,  # Matrix is not square
    # We would need to also implement passing the edge information in order to use
    # preprocessing_edges. As shown later, graph2mat can do this automatically for you.
    preprocessing_edges=None,
)

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/

Now, we can use the matrix model, passing the node features computed by MACE:

In [7]:
node_labels, edge_labels = matrix_model(data=data, node_feats=mace_output["node_feats"])

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0 1 0 1]
shapes:  [[1 4]
 [2 5]]
shapes_inv:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
indices:  [ 0  1  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25  2  3
 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45  4  5 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65]
In Graph2Mat _forward_interactions: 
graph2mat_edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([ True, False,  True, False,  True, False,  True, False])
j_edges: (~i_edges) tensor([False,  True, False,  True, False,  True, False,  True])
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([False,  True

And plot the obtained matrices:

In [8]:
matrices = processor.matrix_from_data(
    data,
    predictions={"node_labels": node_labels, "edge_labels": edge_labels},
)

for config, matrix in zip(configs, matrices):
    plot_basis_matrix(
        matrix,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

In labels_to: of MatrixDataProcessor
data_format: nodesedges
node_labels: [ 4.9140086e-05 -5.6625388e-05 -9.5895249e-03  3.8638224e-03
  0.0000000e+00  0.0000000e+00 -4.6787936e-05  0.0000000e+00
  0.0000000e+00 -2.8328632e-03  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00 -2.8328632e-03
  0.0000000e+00 -2.5948724e-05  6.4659762e-05  0.0000000e+00
  0.0000000e+00 -2.8327708e-03  7.2290059e-05  3.7319842e-07]
edge_labels: [-0.00320521 -0.00164219  0.          0.          0.00028454 -0.00319899
 -0.0016399   0.          0.          0.00077808 -0.00343862 -0.001763
  0.          0.         -0.0010559  -0.00343991 -0.00176355  0.
  0.         -0.00105768  0.00294187 -0.0010882   0.          0.
  0.          0.         -0.00020633  0.00013127  0.00293442 -0.00108501
  0.          0.          0.          0.         -0.00057251  0.00027038
  0.00315525 -0.0011665   0.          0.          0.          0.
  0.00077626 -0.00040401  0.00315489 -0.00116621  0.         

IndexError: boolean index did not match indexed array along dimension 0; dimension is 50 but corresponding boolean dimension is 74

Using MatrixMACE
----------------

If you don't want to handle the details of interacting `MACE` with `Graph2Mat`, you can also use `MatrixMACE`, which takes a mace model and wraps it to also output the `node_labels` and `edge_labels` corresponding to a matrix. 

Internally, it just initializes a `E3nnGraph2Mat` layer. However it can handle the interaction between `MACE` and `Graph2Mat` in more complex cases like having an extra preprocessing step for edges, which needs some extra inputs from MACE.

In [ ]:
from graph2mat.models import MatrixMACE
from graph2mat.bindings.e3nn import E3nnEdgeMessageBlock

In [ ]:
matrix_mace_model = MatrixMACE(
    mace_model,
    unique_basis=table,
    readout_per_interaction=True,
    edge_hidden_irreps=o3.Irreps("10x0e + 10x1o + 10x2e"),
    symmetric=True,
)

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.1

The output of this model is MACE's output plus the `node_labels` and `edge_labels` for the predicted matrix:

In [ ]:
out = matrix_mace_model(data)

out

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0 1 0 1]
shapes:  [[1 5]
 [1 5]]
shapes_inv:  [[1 5]
 [1 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
indices:  [ 0  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25
 26 27  1 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52  2 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77]
In Graph2Mat _forward_interactions: 
graph2mat_edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([ True, False,  True, False,  True, False,  True, False])
j_edges: (~i_edges) tensor([False,  True, False,  True, False,  True, False,  True])
In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resort

{'energy': tensor([-0.0589, -0.0661], grad_fn=<SumBackward1>),
 'node_energy': tensor([-0.0175, -0.0249, -0.0165, -0.0246, -0.0167, -0.0248],
        grad_fn=<SumBackward1>),
 'contributions': tensor([[ 0.0000,  0.0000, -0.5281,  0.5085, -0.0393],
         [ 0.0000,  0.0000, -0.9942,  1.0064, -0.0782]],
        grad_fn=<StackBackward0>),
 'forces': None,
 'edge_forces': None,
 'virials': None,
 'stress': None,
 'atomic_virials': None,
 'atomic_stresses': None,
 'displacement': tensor([[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],
 
         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]),
 'hessian': None,
 'node_feats': tensor([[-2.8360e-02,  0.0000e+00,  0.0000e+00, -4.5525e-03,  2.5390e-03,
           0.0000e+00,  0.0000e+00,  3.4675e-03,  3.5533e-04],
         [-6.7081e-01,  0.0000e+00,  0.0000e+00,  2.0654e-02,  4.2614e-01,
           0.0000e+00,  0.0000e+00, -5.8870e-03, -1.0167e-01],
         [-2.8020e-02,  0.0000e+00,  0.0000e+00, -5.2510e-03,

You can of course plot the predicted matrices:

In [ ]:
matrices = processor.matrix_from_data(data, predictions=out)

for config, matrix in zip(configs, matrices):
    plot_basis_matrix(
        matrix,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

In labels_to: of MatrixDataProcessor
data_format: nodesedges
node_labels: [ 8.6022854e-05 -8.4189691e-02  4.3907105e-03  0.0000000e+00
  0.0000000e+00 -3.8296871e-03  4.3907105e-03 -4.0422704e-02
  0.0000000e+00  0.0000000e+00  9.0410365e-03  0.0000000e+00
  0.0000000e+00  1.5064017e-02  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  1.5064017e-02
  0.0000000e+00 -3.8296871e-03  9.0410365e-03  0.0000000e+00
  0.0000000e+00  1.5133511e-02  7.7517929e-05]
edge_labels: [-2.1509328e-07  1.1787945e-06  0.0000000e+00  0.0000000e+00
  1.1123675e-06 -4.4724252e-06  5.0728263e-06  0.0000000e+00
  0.0000000e+00 -5.3489498e-06]
edge_index: [[0 2]
 [1 1]]
Len rows in _blockmatrix_coo_coords: 47
Len cols in _blockmatrix_coo_coords: 47
Len rows in _nodes_and_edges_to_coo: 47
Len cols in _nodes_and_edges_to_coo: 47
Shape in _nodes_and_edges_to_coo: (7, 7)
Len node_vals in _nodes_and_edges_to_coo: 27
Len edge_vals in _nodes_and_edges_to_coo: 10
Len sparse_data in _nodes_an

Summary and next steps
----------------------

In this notebook we learned **how to interface MACE with Graph2Mat**.

The **next steps** could be:

- **Train a MACE+Graph2Mat model** following the steps in [this notebook](<./Fitting matrices.ipynb>), replacing the model by the `MACE+Graph2Mat` model.